# Landsat Change Detection - Noida

This project compares Landsat 8 images from 13 December 2015 and 30 July 2024.

The steps are image preprocessing, MLE classification, linear regression and change classification.


## 1. Install packages

In [1]:
%pip -q install rasterio numpy matplotlib scikit-learn

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


## 2. Connect Google Drive

Keep all ten TIF files in one folder and enter that folder below.


In [2]:
from pathlib import Path
import sys

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    drive_root = Path("/content/drive/MyDrive")
    candidates = list(drive_root.rglob("LC08_L2SP_146040_20151213*_SR_B2.TIF"))
    if not candidates:
        raise FileNotFoundError(
            "The Landsat files were not found in Google Drive. "
            "Upload all 10 TIF files to one folder in My Drive, then run this cell again."
        )
    DATA_DIR = candidates[0].parent
else:
    DATA_DIR = Path(r"C:\Users\saini\Downloads\ml datasets -20260920T193242Z-1-001\ml datasets")
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Data folder not found: {DATA_DIR}")
print("Data folder:", DATA_DIR)


Data folder: C:\Users\saini\Downloads\ml datasets -20260920T193242Z-1-001\ml datasets


## 3. Imports and file discovery

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject
from sklearn.linear_model import LinearRegression

DATES = {"2015": "20151213", "2024": "20240730"}
BANDS = ("SR_B2", "SR_B3", "SR_B4", "SR_B5")
BAD_QA_BITS = sum(1 << bit for bit in range(6))
SCALE = 0.0000275
OFFSET = -0.2
GRID_FACTOR = 5  # 30 m source -> 150 m demonstration grid
RNG = np.random.default_rng(42)

def locate(date_token, band):
    matches = sorted(DATA_DIR.glob(f"LC08_L2SP_*_{date_token}_*_{band}.TIF"))
    if len(matches) != 1:
        raise ValueError(f"Expected one {date_token} {band} file, found {len(matches)}")
    return matches[0]

FILES = {year: {band: locate(token, band) for band in (*BANDS, "QA_PIXEL")}
         for year, token in DATES.items()}

for year, items in FILES.items():
    print(f"{year}:")
    for band, path in items.items():
        print(f"  {band}: {path.name}")


2015:
  SR_B2: LC08_L2SP_146040_20151213_20200908_02_T1_SR_B2.TIF
  SR_B3: LC08_L2SP_146040_20151213_20200908_02_T1_SR_B3.TIF
  SR_B4: LC08_L2SP_146040_20151213_20200908_02_T1_SR_B4.TIF
  SR_B5: LC08_L2SP_146040_20151213_20200908_02_T1_SR_B5.TIF
  QA_PIXEL: LC08_L2SP_146040_20151213_20200908_02_T1_QA_PIXEL.TIF
2024:
  SR_B2: LC08_L2SP_146040_20240730_20240801_02_T1_SR_B2.TIF
  SR_B3: LC08_L2SP_146040_20240730_20240801_02_T1_SR_B3.TIF
  SR_B4: LC08_L2SP_146040_20240730_20240801_02_T1_SR_B4.TIF
  SR_B5: LC08_L2SP_146040_20240730_20240801_02_T1_SR_B5.TIF
  QA_PIXEL: LC08_L2SP_146040_20240730_20240801_02_T1_QA_PIXEL.TIF


## 4. Preprocess the images

This cell converts DN values to reflectance, removes bad pixels and places both dates on the same grid.


In [4]:
def make_reference_grid():
    with rasterio.open(FILES["2015"]["SR_B4"]) as src:
        height = (src.height + GRID_FACTOR - 1) // GRID_FACTOR
        width = (src.width + GRID_FACTOR - 1) // GRID_FACTOR
        transform = src.transform * src.transform.scale(src.width / width, src.height / height)
        return {"crs": src.crs, "transform": transform, "height": height,
                "width": width, "source_shape": (src.height, src.width),
                "resolution": src.res, "bounds": src.bounds}

TARGET = make_reference_grid()

def read_to_reference(path):
    output = np.zeros((TARGET["height"], TARGET["width"]), dtype=np.float32)
    with rasterio.open(path) as src:
        reproject(rasterio.band(src, 1), output,
                  src_transform=src.transform, src_crs=src.crs, src_nodata=src.nodata,
                  dst_transform=TARGET["transform"], dst_crs=TARGET["crs"], dst_nodata=0,
                  resampling=Resampling.nearest)
    return output

def load_year(year):
    qa = read_to_reference(FILES[year]["QA_PIXEL"]).astype(np.uint16)
    qa_valid = (qa & BAD_QA_BITS) == 0
    layers = []
    for band in BANDS:
        dn = read_to_reference(FILES[year][band])
        valid = qa_valid & (dn > 0)
        reflectance = dn * np.float32(SCALE) + np.float32(OFFSET)
        reflectance[~valid] = np.nan
        layers.append(reflectance)
    stack = np.stack(layers)
    return stack, np.isfinite(stack).all(axis=0)

image_2015, valid_2015 = load_year("2015")
image_2024, valid_2024 = load_year("2024")
common_valid = valid_2015 & valid_2024

print("CRS:", TARGET["crs"])
print("Native shape:", TARGET["source_shape"])
print("Native pixel size:", TARGET["resolution"])
print("Analysis shape:", common_valid.shape)
print("Common valid analysis pixels:", f"{common_valid.sum():,}")


RasterioIOError: LC08_L2SP_146040_20151213_20200908_02_T1_SR_B4.TIF: MissingRequired:TIFF directory is missing required "StripByteCounts" field

## 5. Natural-colour visualization

In [ ]:
def rgb(stack, valid):
    image = np.moveaxis(stack[[2, 1, 0]], 0, -1)  # B4, B3, B2
    image = np.clip(np.nan_to_num(image) / 0.30, 0, 1)
    return np.dstack([image, valid.astype(np.float32)])

fig, axes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)
axes[0].imshow(rgb(image_2015, common_valid))
axes[0].set_title("2015 natural colour")
axes[1].imshow(rgb(image_2024, common_valid))
axes[1].set_title("2024 natural colour")
for ax in axes:
    ax.axis("off")
plt.show()


## 6. Maximum Likelihood Estimation

B2, B3, B4 and B5 are used as features. Training samples are selected using simple spectral rules for this demonstration.


In [ ]:
def create_seed_masks(stack, valid):
    blue, green, red, nir = stack
    ndvi = (nir - red) / (nir + red + 1e-6)
    ndwi = (green - nir) / (green + nir + 1e-6)
    visible = stack[:3]
    finite = np.isfinite(visible)
    brightness = np.divide(np.nansum(visible, axis=0), finite.sum(axis=0),
                           out=np.full(valid.shape, np.nan, dtype=np.float32),
                           where=finite.sum(axis=0) > 0)
    nq = np.quantile(ndvi[valid], [0.30, 0.55, 0.80])
    wq = np.quantile(ndwi[valid], 0.90)
    bq = np.quantile(brightness[valid], [0.45, 0.75])
    return {
        "Water": valid & (ndwi >= wq) & (ndvi <= nq[0]),
        "Vegetation": valid & (ndvi >= nq[2]),
        "Built-up": valid & (brightness >= bq[1]) & (ndvi <= nq[1]),
        "Bare Land": valid & (brightness >= bq[0]) & (ndvi > nq[0]) & (ndvi < nq[2]),
    }

def train_mle(stack, masks):
    pixels = np.moveaxis(stack, 0, -1)
    models = []
    for class_id, (name, mask) in enumerate(masks.items()):
        samples = pixels[mask]
        print(f"{name}: {len(samples):,} provisional samples")
        if len(samples) > 5000:
            samples = samples[RNG.choice(len(samples), 5000, replace=False)]
        covariance = np.cov(samples, rowvar=False) + np.eye(4) * 1e-6
        models.append((class_id, name, samples.mean(axis=0),
                       np.linalg.pinv(covariance), np.linalg.slogdet(covariance)[1]))
    return models

def mle_predict(stack, valid, models):
    pixels = np.moveaxis(stack, 0, -1)
    positions = np.flatnonzero(valid)
    x = pixels.reshape(-1, 4)[positions]
    best = np.full(len(x), -np.inf)
    labels = np.full(len(x), -1, dtype=np.int8)
    for class_id, name, mean, inverse_covariance, log_determinant in models:
        difference = x - mean
        score = -0.5 * (log_determinant + np.einsum(
            "ij,jk,ik->i", difference, inverse_covariance, difference))
        select = score > best
        labels[select], best[select] = class_id, score[select]
    result = np.full(valid.shape, -1, dtype=np.int8)
    result.flat[positions] = labels
    return result

seed_masks = create_seed_masks(image_2015, valid_2015)
mle_models = train_mle(image_2015, seed_masks)
mle_2015 = mle_predict(image_2015, valid_2015, mle_models)
mle_2024 = mle_predict(image_2024, valid_2024, mle_models)
print("MLE classification completed.")


## 7. Linear regression and change classification

Regression compares matching pixels from 2015 and 2024. Large residuals are marked as possible change.


In [ ]:
old_pixels = np.moveaxis(image_2015, 0, -1)[common_valid]
new_pixels = np.moveaxis(image_2024, 0, -1)[common_valid]
sample_count = min(100_000, len(old_pixels))
sample_indices = RNG.choice(len(old_pixels), sample_count, replace=False)
residuals = np.empty_like(new_pixels, dtype=np.float32)
regression_results = {}

for index, band in enumerate(BANDS):
    model = LinearRegression().fit(old_pixels[sample_indices, index, None],
                                   new_pixels[sample_indices, index])
    prediction = model.predict(old_pixels[:, index, None])
    residuals[:, index] = new_pixels[:, index] - prediction
    regression_results[band] = {
        "slope": model.coef_[0], "intercept": model.intercept_,
        "r2": model.score(old_pixels[sample_indices, index, None],
                          new_pixels[sample_indices, index]),
        "rmse": np.sqrt(np.mean(residuals[:, index] ** 2)),
    }

print(f"{'Band':<8} {'Slope':>9} {'Intercept':>11} {'R2':>9} {'RMSE':>9}")
for band, result in regression_results.items():
    print(f"{band:<8} {result['slope']:>9.4f} {result['intercept']:>11.4f} "
          f"{result['r2']:>9.4f} {result['rmse']:>9.4f}")

residual_scale = np.median(np.abs(residuals), axis=0) + 1e-6
scores = np.sqrt(np.mean((residuals / residual_scale) ** 2, axis=1))
score_median = np.median(scores)
score_mad = np.median(np.abs(scores - score_median))
threshold = score_median + 3 * 1.4826 * score_mad
possible_change = scores > threshold

score_map = np.full(common_valid.shape, np.nan, dtype=np.float32)
change_map = np.full(common_valid.shape, -1, dtype=np.int8)
score_map[common_valid] = scores
change_map[common_valid] = possible_change.astype(np.int8)

print("Residual threshold:", round(float(threshold), 4))
print("Possible-change pixels:", f"{possible_change.sum():,}")
print("Possible-change percentage:", f"{100 * possible_change.mean():.2f}%")


## 8. Final Matplotlib results

In [ ]:
from matplotlib.colors import ListedColormap

class_names = [model[1] for model in mle_models]
class_cmap = ListedColormap(["#2b83ba", "#1a9850", "#d8b365", "#a6611a"])
change_cmap = ListedColormap(["#202020", "#e31a1c"])
change_cmap.set_bad("#d9d9d9")

fig, axes = plt.subplots(2, 3, figsize=(17, 10), constrained_layout=True)
axes[0, 0].imshow(rgb(image_2015, common_valid)); axes[0, 0].set_title("2015 natural colour")
axes[0, 1].imshow(rgb(image_2024, common_valid)); axes[0, 1].set_title("2024 natural colour")
im_mle = axes[0, 2].imshow(np.ma.masked_less(mle_2015, 0), cmap=class_cmap, vmin=0, vmax=3)
axes[0, 2].set_title("2015 MLE Classification")
axes[1, 0].imshow(np.ma.masked_less(mle_2024, 0), cmap=class_cmap, vmin=0, vmax=3)
axes[1, 0].set_title("2024 MLE Classification")
im_score = axes[1, 1].imshow(score_map, cmap="magma", vmin=0, vmax=np.nanpercentile(score_map, 98))
axes[1, 1].set_title("Regression residual score")
fig.colorbar(im_score, ax=axes[1, 1], shrink=0.75)
axes[1, 2].imshow(np.ma.masked_less(change_map, 0), cmap=change_cmap, vmin=0, vmax=1)
axes[1, 2].set_title("Possible change (red); invalid (gray)")
for ax in axes.flat:
    ax.axis("off")
cbar = fig.colorbar(im_mle, ax=[axes[0, 2], axes[1, 0]], ticks=range(4), shrink=0.75)
cbar.ax.set_yticklabels(class_names)
fig.suptitle("Landsat coding challenge results", fontsize=15)
plt.savefig("landsat_colab_results.png", dpi=160, bbox_inches="tight")
plt.show()
print("Saved: landsat_colab_results.png")


## 9. Note

The 2015 image is from December and the 2024 image is from July, so some differences may be seasonal. The dataset does not include ground-truth labels, so the notebook does not calculate final model accuracy.
